# KV Cache With A Real Hugging Face Model

This notebook keeps the thing you wanted to inspect: `AutoModelForCausalLM.generate(..., use_cache=True)`.

The original code used `.cuda()`. That only works on an NVIDIA GPU. This notebook chooses `cuda`, `mps`, or `cpu` automatically, then shows the actual `past_key_values` tensors used by the KV cache.

In [1]:
import sys
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, LlamaConfig, LlamaForCausalLM

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("LlamaConfig import OK:", LlamaConfig.__name__)
print("LlamaForCausalLM import OK:", LlamaForCausalLM.__name__)

if torch.cuda.is_available():
    device = "cuda"
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

# float16 is usually best on CUDA. CPU is safest with float32.
dtype = torch.float16 if device == "cuda" else torch.float32

print("Device:", device)
print("Model dtype:", dtype)

/Users/deveshi/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python executable: /usr/local/bin/python3
Python version: 3.9.13 (v3.9.13:6de2ca5339, May 17 2022, 11:37:23) 
[Clang 13.0.0 (clang-1300.0.29.30)]
PyTorch version: 2.4.0
LlamaConfig import OK: LlamaConfig
LlamaForCausalLM import OK: LlamaForCausalLM
Device: mps
Model dtype: torch.float32


## Choose A Model

Your original model is `HuggingFaceTB/SmolLM2-1.7B`. It may be slow or memory-heavy on CPU.

If your laptop struggles, try the smaller `HuggingFaceTB/SmolLM2-135M` first. The KV-cache mechanics are the same; the tensors are just smaller.

In [9]:
# model_id = "HuggingFaceTB/SmolLM2-1.7B"
model_id = "HuggingFaceTB/SmolLM2-135M"

model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=dtype)
model = model.to(device).eval()

print("Loaded:", model_id)
print("Number of layers:", getattr(model.config, "num_hidden_layers", None))
print("Attention heads:", getattr(model.config, "num_attention_heads", None))
print("KV heads:", getattr(model.config, "num_key_value_heads", getattr(model.config, "num_attention_heads", None)))
print("Hidden size:", getattr(model.config, "hidden_size", None))

Loaded: HuggingFaceTB/SmolLM2-135M
Number of layers: 30
Attention heads: 9
KV heads: 3
Hidden size: 576


## Your Original Generate Call

This is your code, but without hardcoded `.cuda()`. `use_cache=True` is the default for generation, but it is written explicitly here.

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

inputs = tokenizer("Once upon a", return_tensors="pt").to(device)
print("Prompt:", tokenizer.batch_decode(inputs.input_ids, skip_special_tokens=True)[0])

def first_layer_kv(past_key_values):
    # Works for tuple-style cache and newer HF Cache objects.
    if hasattr(past_key_values, "key_cache"):
        return past_key_values.key_cache[0], past_key_values.value_cache[0]

    layer0 = past_key_values[0]
    return layer0[0], layer0[1]

def cache_nbytes(past_key_values):
    if hasattr(past_key_values, "key_cache"):
        pairs = zip(past_key_values.key_cache, past_key_values.value_cache)
    else:
        pairs = [(layer[0], layer[1]) for layer in past_key_values]

    total = 0
    for key, value in pairs:
        total += key.numel() * key.element_size()
        total += value.numel() * value.element_size()
    return total

def show_cache(label, past_key_values, token_text=None):
    key, value = first_layer_kv(past_key_values)
    cached_tokens = key.shape[-2]

    msg = (
        f"{label:>8} | "
        f"cached_tokens={cached_tokens:>4} | "
        f"layer0 K={tuple(key.shape)} | "
        f"layer0 V={tuple(value.shape)} | "
        f"cache={cache_nbytes(past_key_values) / 1024**2:.2f} MiB"
    )

    if token_text is not None:
        msg += f" | token={token_text!r}"

    print(msg)

max_new_tokens = 100

generated = inputs["input_ids"].clone()
attention_mask = inputs["attention_mask"].clone()

with torch.no_grad():
    # 1. Prefill: process the whole prompt once.
    outputs = model(
        input_ids=generated,
        attention_mask=attention_mask,
        use_cache=True,
    )

    past_key_values = outputs.past_key_values
    show_cache("prompt", past_key_values)

    # 2. Decode: generate one token at a time.
    for step in range(1, max_new_tokens + 1):
        next_token = outputs.logits[:, -1, :].argmax(dim=-1, keepdim=True)

        token_id = next_token[0, 0].item()
        token_text = tokenizer.decode([token_id])

        generated = torch.cat([generated, next_token], dim=-1)

        # MPS needs attention_mask explicitly. It grows with the sequence.
        attention_mask = torch.cat(
            [attention_mask, torch.ones_like(next_token)],
            dim=-1,
        )

        outputs = model(
            input_ids=next_token,
            attention_mask=attention_mask,
            past_key_values=past_key_values,
            use_cache=True,
        )

        past_key_values = outputs.past_key_values

        # This prints every decoded token. For 2000 tokens, expect a lot of output.
        show_cache(f"step {step}", past_key_values, token_text=token_text)

output_text = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
print("\nFinal output:")
print(output_text)


Prompt: Once upon a
  prompt | cached_tokens=   3 | layer0 K=(1, 3, 3, 64) | layer0 V=(1, 3, 3, 64) | cache=0.13 MiB
  step 1 | cached_tokens=   4 | layer0 K=(1, 3, 4, 64) | layer0 V=(1, 3, 4, 64) | cache=0.18 MiB | token=' time'
  step 2 | cached_tokens=   5 | layer0 K=(1, 3, 5, 64) | layer0 V=(1, 3, 5, 64) | cache=0.22 MiB | token=','
  step 3 | cached_tokens=   6 | layer0 K=(1, 3, 6, 64) | layer0 V=(1, 3, 6, 64) | cache=0.26 MiB | token=' there'
  step 4 | cached_tokens=   7 | layer0 K=(1, 3, 7, 64) | layer0 V=(1, 3, 7, 64) | cache=0.31 MiB | token=' was'
  step 5 | cached_tokens=   8 | layer0 K=(1, 3, 8, 64) | layer0 V=(1, 3, 8, 64) | cache=0.35 MiB | token=' a'
  step 6 | cached_tokens=   9 | layer0 K=(1, 3, 9, 64) | layer0 V=(1, 3, 9, 64) | cache=0.40 MiB | token=' little'
  step 7 | cached_tokens=  10 | layer0 K=(1, 3, 10, 64) | layer0 V=(1, 3, 10, 64) | cache=0.44 MiB | token=' girl'
  step 8 | cached_tokens=  11 | layer0 K=(1, 3, 11, 64) | layer0 V=(1, 3, 11, 64) | cache=0.48 

## Inspect The KV Cache Directly

`generate()` hides the cache loop from you. To see the cache clearly, run the model manually once on the full prompt. The returned `past_key_values` is the KV cache.

Each layer stores one key tensor and one value tensor.

## Decode One Token And Watch Cache Grow

During generation, the model does not feed the whole prompt again. It feeds only the newest token plus the previous cache.

This is the core optimization.

In [13]:
max_new_tokens = 20

# Make this cell self-contained instead of depending on a previous `tokens` variable.

prompt = "The red cat was"
manual_inputs = tokenizer(prompt, return_tensors="pt").to(device)
generated = manual_inputs["input_ids"].clone()

with torch.no_grad():
    # Prefill: run the whole prompt once and create the initial KV cache.
    outputs = model(
        input_ids=generated,
        use_cache=True,
    )
    past_key_values = outputs.past_key_values
    print(past_key_values[0][0].shape if isinstance(past_key_values[0][0], torch.Tensor) else "past_key_values[0][0] is not a tensor")
    print(outputs.logits.shape if isinstance(outputs.logits, torch.Tensor) else "outputs.logits is not a tensor")

    next_token = outputs.logits[:, -1:].argmax(dim=-1)
    token_id = next_token[0, 0].item()
    token_text = tokenizer.decode([token_id])
    print(f"{token_id} -> {token_text!r}")


    for step in range(max_new_tokens):
        generated = torch.cat([generated, next_token], dim=-1)

        # Decode: feed only the newest token plus the old KV cache.
        outputs = model(
            input_ids=next_token,
            past_key_values=past_key_values,
            use_cache=True,
        )

        past_key_values = outputs.past_key_values
        next_token = outputs.logits[:, -1:].argmax(dim=-1)

        if step in [0, 1, 2, max_new_tokens - 1]:
            show_cache(f"After manual decode step {step + 1}", past_key_values)

print(tokenizer.batch_decode(generated, skip_special_tokens=True)[0])

torch.Size([1, 3, 4, 64])
torch.Size([1, 4, 49152])
253 -> ' a'
After manual decode step 1 | cached_tokens=   5 | layer0 K=(1, 3, 5, 64) | layer0 V=(1, 3, 5, 64) | cache=0.22 MiB
After manual decode step 2 | cached_tokens=   6 | layer0 K=(1, 3, 6, 64) | layer0 V=(1, 3, 6, 64) | cache=0.26 MiB
After manual decode step 3 | cached_tokens=   7 | layer0 K=(1, 3, 7, 64) | layer0 V=(1, 3, 7, 64) | cache=0.31 MiB
After manual decode step 20 | cached_tokens=  24 | layer0 K=(1, 3, 24, 64) | layer0 V=(1, 3, 24, 64) | cache=1.05 MiB
The red cat was a symbol of the
olden times, and the white cat was a symbol of the new.


## Compare `use_cache=True` vs `use_cache=False`

This rough timing cell shows why the cache matters. With `use_cache=False`, generation has to redo much more attention work each step.

In [12]:
def sync_device():
    if device == "cuda":
        torch.cuda.synchronize()
    elif device == "mps":
        torch.mps.synchronize()

def timed_generate(use_cache, max_new_tokens=100, warmup=True):
    if warmup:
        with torch.no_grad():
            _ = model.generate(
                **inputs,
                max_new_tokens=5,
                use_cache=use_cache,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        sync_device()

    start = time.perf_counter()
    with torch.no_grad():
        _ = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            use_cache=use_cache,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    sync_device()
    return time.perf_counter() - start

print("cache on: ", timed_generate(True, max_new_tokens=200))
print("cache off:", timed_generate(False, max_new_tokens=200))


cache on:  41.78960024999992
cache off: 128.23479895800006


## What To Change

- `model_id`: switch between `SmolLM2-1.7B` and `SmolLM2-135M`.
- `prompt`: longer prompts create a bigger initial cache.
- `max_new_tokens`: more generated tokens make the cache grow longer.
- `use_cache`: compare speed and behavior with `True` vs `False`.
- `dtype`: on CUDA, `float16` cuts cache memory roughly in half compared with `float32`.